# Laboratorio 02 — JOINs en SQL sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 02  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica los tipos de JOIN y subconsultas de la Actividad 02 sobre **dos o más tablas de tu elección** en Databricks. Si tu dataset es una sola tabla, diseña un escenario donde puedas dividirla en dos tablas que tengan sentido relacionar, o agrega una tabla de referencia (p.ej., un catálogo de categorías).

## Parte 1 — Descripción del escenario

1. **Tabla A:** nombre, fuente, qué representa cada fila, columna(s) que usarás para el JOIN.
2. **Tabla B:** nombre, fuente, qué representa cada fila, columna(s) que usarás para el JOIN.
3. **Relación esperada:** ¿Es 1:1, 1:N, N:M? ¿Por qué elegiste estas tablas?
4. **Preguntas de negocio:** Al menos 3 preguntas que requieran combinar las tablas.

**Escribe tu respuesta aquí:**

## Parte 2 — Cargar las tablas como Delta

In [ ]:
VOL     = "/Volumes/workspace/default/week_3"  # ajusta si usas otro volumen
TABLA_A = "workspace.default.lab03_02_tabla_a"
TABLA_B = "workspace.default.lab03_02_tabla_b"

# Carga Tabla A
df_a = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load(f"{VOL}/archivo_a.csv")  # reemplaza con el nombre real
df_a.write.format("delta").mode("overwrite").saveAsTable(TABLA_A)
print(f"✓ Tabla A: {TABLA_A} — {df_a.count():,} filas x {len(df_a.columns)} columnas")

# Carga Tabla B
df_b = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load(f"{VOL}/archivo_b.csv")  # reemplaza con el nombre real
df_b.write.format("delta").mode("overwrite").saveAsTable(TABLA_B)
print(f"✓ Tabla B: {TABLA_B} — {df_b.count():,} filas x {len(df_b.columns)} columnas")

## Parte 3 — Perfil técnico de las tablas

In [ ]:
# Schema de Tabla A
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA_A}").show(20, truncate=False)

In [ ]:
# Schema de Tabla B
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA_B}").show(20, truncate=False)

In [ ]:
# Diagnóstico de duplicados en la clave de JOIN — Tabla A
# Reemplaza 'clave_join_a' con el nombre real de la columna
spark.sql(f"""
    SELECT clave_join_a, COUNT(*) AS apariciones
    FROM {TABLA_A}
    GROUP BY clave_join_a
    HAVING COUNT(*) > 1
    ORDER BY apariciones DESC
    LIMIT 10
""").show(truncate=False)

**Observación clave de JOIN:** ¿Hay duplicados en la clave? ¿Cómo afecta esto al número de filas del resultado del JOIN?

In [ ]:
# Diagnóstico de duplicados en la clave de JOIN — Tabla B
spark.sql(f"""
    SELECT clave_join_b, COUNT(*) AS apariciones
    FROM {TABLA_B}
    GROUP BY clave_join_b
    HAVING COUNT(*) > 1
    ORDER BY apariciones DESC
    LIMIT 10
""").show(truncate=False)

## Parte 4 — Aplicar JOINs de la Actividad 02

Para cada tipo de JOIN muestra también el número de filas resultantes y explica la diferencia en markdown.

In [ ]:
# INNER JOIN: solo registros que coinciden en ambas tablas
resultado_inner = spark.sql(f"""
    SELECT a.*, b.*  -- limita las columnas que necesites
    FROM {TABLA_A} a
    INNER JOIN {TABLA_B} b
        ON a.clave_join_a = b.clave_join_b
    LIMIT 20
""")
resultado_inner.show(truncate=False)
print(f"Filas INNER JOIN: {resultado_inner.count():,}")

**Análisis INNER JOIN:** ¿Cuántos registros de la Tabla A no encontraron coincidencia? ¿Eso representa un problema de calidad?

In [ ]:
# LEFT JOIN: todos los de Tabla A, coincidentes o no con Tabla B
resultado_left = spark.sql(f"""
    SELECT a.*, b.clave_join_b  -- añade las columnas de B que necesites
    FROM {TABLA_A} a
    LEFT JOIN {TABLA_B} b
        ON a.clave_join_a = b.clave_join_b
""")
print(f"Filas LEFT JOIN (total):     {resultado_left.count():,}")
print(f"Filas sin match en B: {resultado_left.filter('clave_join_b IS NULL').count():,}")

**Análisis LEFT JOIN:** Los registros con `clave_join_b IS NULL` son los que no tienen correspondencia. ¿Qué significan en el contexto de negocio?

In [ ]:
# FULL OUTER JOIN: todos los de A y B aunque no coincidan
resultado_full = spark.sql(f"""
    SELECT
        COALESCE(a.clave_join_a, b.clave_join_b) AS clave,
        CASE WHEN a.clave_join_a IS NULL THEN 'Solo en B'
             WHEN b.clave_join_b IS NULL THEN 'Solo en A'
             ELSE 'En ambas' END AS origen
    FROM {TABLA_A} a
    FULL OUTER JOIN {TABLA_B} b
        ON a.clave_join_a = b.clave_join_b
""")
resultado_full.groupBy("origen").count().show()

**Análisis FULL OUTER JOIN:** ¿Cuántos registros son exclusivos de cada tabla? ¿Qué indica eso sobre la integridad referencial?

In [ ]:
# Subconsulta correlacionada: obtener el máximo/mínimo de B para cada registro de A
spark.sql(f"""
    SELECT
        a.*,
        (
            SELECT COUNT(*)
            FROM {TABLA_B} b
            WHERE b.clave_join_b = a.clave_join_a
        ) AS conteo_en_b
    FROM {TABLA_A} a
    ORDER BY conteo_en_b DESC
    LIMIT 15
""").show(truncate=False)

**Análisis subconsulta:** ¿Qué ventaja tiene la subconsulta sobre un JOIN + GROUP BY en este caso? ¿Cuándo podría ser menos eficiente?

## Parte 5 — Preguntas de negocio con JOINs

Responde las 3 preguntas de negocio planteadas en la Parte 1 usando los JOINs adecuados.

In [ ]:
# Pregunta de negocio 1  (usa INNER, LEFT, FULL OUTER o subconsulta según convenga)
spark.sql(f"""

""").show(truncate=False)

**Conclusión pregunta 1:**

In [ ]:
# Pregunta de negocio 2
spark.sql(f"""

""").show(truncate=False)

**Conclusión pregunta 2:**

In [ ]:
# Pregunta de negocio 3
spark.sql(f"""

""").show(truncate=False)

**Conclusión pregunta 3:**

## Parte 6 — Reflexión final

1. ¿Qué tipo de JOIN perdió más registros? ¿Era lo esperado?
2. ¿El número de filas del INNER JOIN fue mayor, igual o menor a cualquiera de las tablas originales? ¿Por qué?
3. ¿En qué situación de tu dominio elegirías un FULL OUTER JOIN sobre un LEFT JOIN?
4. ¿Cómo expresarías la subconsulta correlacionada usando PySpark? ¿Sería más sencillo o más complejo?

---

## Entrega en Git

```bash
git add semana_03/laboratorios/lab_02_sql_joins.ipynb
git commit -m "lab: semana03 lab02 sql joins <nombre-dataset> - <tu-nombre>"
git push origin feature/semana03-sql-<tu-nombre>
```